<a href="https://colab.research.google.com/github/Seif-Elmezaien/Amit-AI-Course/blob/Python/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-Elmezaien/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Rule

I will prioritize pages for refresh when they show signs of being **stale and having weak search performance**. The baseline score will combine content freshness with search performance signals, using only information available at the time of the decision.

Pages with higher scores will receive higher refresh priority.

### Reason codes

- `STALE` — the content has not been updated recently.
- `WEAK_SEARCH_PERFORMANCE` — the page shows weak search performance based on its search signal.
- `STALE_AND_WEAK` — the page is both stale and has weak search performance.
- `NO_STRONG_SIGNAL` — the page does not meet either condition.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
import numpy as np
import pandas as pd

baseline = df.copy()

# -----------------------------
# 1. Create the two signals
# -----------------------------

# Low CTR: lowest observed CTR bucket from the signal audit
baseline["weak_ctr"] = baseline["ctr"] < 0.14

# Stale content: 180+ days since last update
baseline["stale"] = baseline["days_since_last_update"] >= 180


# -----------------------------
# 2. Calculate baseline score
# -----------------------------

baseline["score"] = (
    baseline["weak_ctr"].astype(int)
    + baseline["stale"].astype(int)
)


# -----------------------------
# 3. Assign one reason code
# -----------------------------

def get_reason_code(row):
    if row["weak_ctr"] and row["stale"]:
        return "WEAK_CTR_AND_STALE"
    elif row["weak_ctr"]:
        return "WEAK_CTR"
    elif row["stale"]:
        return "STALE"
    else:
        return "NO_STRONG_SIGNAL"

baseline["reason_code"] = baseline.apply(
    get_reason_code,
    axis=1
)


# -----------------------------
# 4. Assign action
# -----------------------------

baseline["action"] = np.select(
    [
        baseline["score"] == 2,
        baseline["score"] == 1
    ],
    [
        "REFRESH",
        "REVIEW"
    ],
    default="MONITOR"
)


# -----------------------------
# 5. Rank the queue
# -----------------------------

baseline = baseline.sort_values(
    by=["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1


# -----------------------------
# 6. Select output columns
# -----------------------------

output = baseline[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action"
    ]
]


# -----------------------------
# 7. Write the ranked queue
# -----------------------------

# output.to_csv(
#     "work/outputs/baseline_action_score.csv",
#     index=False
# )

output.head(10)

,rank,content_id,client_id,score,reason_code,action
0,1,content_7368877ea310,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH
1,2,content_5feee3994adb,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH
2,3,content_b16bd7307b39,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH
3,4,content_928af3e22c80,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH
4,5,content_074ba6ead17b,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH
5,6,content_fd16e3475c29,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH
6,7,content_6476d1d8c050,client_19581e27de,2,WEAK_CTR_AND_STALE,REFRESH
7,8,content_4f241bad48a3,client_6208ef0f77,2,WEAK_CTR_AND_STALE,REFRESH
8,9,content_ea41fe5cf292,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH
9,10,content_d25a099b3726,client_19581e27de,2,WEAK_CTR_AND_STALE,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the twenty highest-ranked pages from the baseline queue.
The confidence note reflects how many of the rule's signals are present,
while the "what would make it wrong" note identifies limitations that could
make the recommendation misleading.

In [12]:
top20 = baseline.head(20).copy()

top20_review = top20[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "ctr",
        "days_since_last_update",
        "impressions_90d",
        "avg_position"
    ]
].copy()

top20_review


,rank,content_id,client_id,score,reason_code,action,ctr,days_since_last_update,impressions_90d,avg_position
0,1,content_7368877ea310,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.13,194,59472,24.8
1,2,content_5feee3994adb,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.01,194,7812,39.0
2,3,content_b16bd7307b39,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.00,194,4590,31.0
3,4,content_928af3e22c80,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.12,193,1697,15.8
4,5,content_074ba6ead17b,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,533,48.0
5,6,content_fd16e3475c29,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,429,9.0
6,7,content_6476d1d8c050,client_19581e27de,2,WEAK_CTR_AND_STALE,REFRESH,0.00,313,304,67.8
7,8,content_4f241bad48a3,client_6208ef0f77,2,WEAK_CTR_AND_STALE,REFRESH,0.00,236,285,19.1
8,9,content_ea41fe5cf292,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,265,8.0
9,10,content_d25a099b3726,client_19581e27de,2,WEAK_CTR_AND_STALE,REFRESH,0.00,305,202,64.5


In [13]:
def confidence_note(row):
    if row["score"] == 2:
        return "Higher confidence: both signals are present."
    elif row["score"] == 1:
        return "Moderate confidence: only one signal is present."
    else:
        return "Low confidence: no strong signal."


def what_would_make_it_wrong(row):
    reasons = []

    if row["impressions_90d"] < 100:
        reasons.append("very low impressions make CTR noisy")

    if row["reason_code"] == "STALE":
        reasons.append("staleness alone does not prove declining performance")

    if row["reason_code"] == "WEAK_CTR":
        reasons.append("CTR depends strongly on search position")

    if row["reason_code"] == "WEAK_CTR_AND_STALE":
        reasons.append("low CTR may be explained by search position or low volume")

    return "; ".join(reasons)


top20_review["confidence_note"] = top20_review.apply(
    confidence_note,
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    what_would_make_it_wrong,
    axis=1
)

top20_review

,rank,content_id,client_id,score,reason_code,action,ctr,days_since_last_update,impressions_90d,avg_position,confidence_note,what_would_make_it_wrong
0,1,content_7368877ea310,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.13,194,59472,24.8,Higher confidence: both signals are present.,low CTR may be explained by search position or...
1,2,content_5feee3994adb,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.01,194,7812,39.0,Higher confidence: both signals are present.,low CTR may be explained by search position or...
2,3,content_b16bd7307b39,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.00,194,4590,31.0,Higher confidence: both signals are present.,low CTR may be explained by search position or...
3,4,content_928af3e22c80,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.12,193,1697,15.8,Higher confidence: both signals are present.,low CTR may be explained by search position or...
4,5,content_074ba6ead17b,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,533,48.0,Higher confidence: both signals are present.,low CTR may be explained by search position or...
5,6,content_fd16e3475c29,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,429,9.0,Higher confidence: both signals are present.,low CTR may be explained by search position or...
6,7,content_6476d1d8c050,client_19581e27de,2,WEAK_CTR_AND_STALE,REFRESH,0.00,313,304,67.8,Higher confidence: both signals are present.,low CTR may be explained by search position or...
7,8,content_4f241bad48a3,client_6208ef0f77,2,WEAK_CTR_AND_STALE,REFRESH,0.00,236,285,19.1,Higher confidence: both signals are present.,low CTR may be explained by search position or...
8,9,content_ea41fe5cf292,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,265,8.0,Higher confidence: both signals are present.,low CTR may be explained by search position or...
9,10,content_d25a099b3726,client_19581e27de,2,WEAK_CTR_AND_STALE,REFRESH,0.00,305,202,64.5,Higher confidence: both signals are present.,low CTR may be explained by search position or...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Leakage check

The baseline uses only information available at the decision time:
`ctr`, `days_since_last_update`, and `impressions_90d`.

I did not use `trend_pct`, `trend_direction`, or `is_declining_label`,
because these are label-derived signals.

I also did not use any future-window impressions, clicks, sessions,
or other future performance measurements.

The baseline is therefore intended to represent a rule that could
actually be applied at the decision time.

In [14]:
# Show the top 20 baseline picks with the signals
weak_pick_candidates = baseline[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "ctr",
        "days_since_last_update",
        "impressions_90d",
        "avg_position"
    ]
].head(20)

weak_pick_candidates


,rank,content_id,client_id,score,reason_code,action,ctr,days_since_last_update,impressions_90d,avg_position
0,1,content_7368877ea310,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.13,194,59472,24.8
1,2,content_5feee3994adb,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.01,194,7812,39.0
2,3,content_b16bd7307b39,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.00,194,4590,31.0
3,4,content_928af3e22c80,client_7f2253d7e2,2,WEAK_CTR_AND_STALE,REFRESH,0.12,193,1697,15.8
4,5,content_074ba6ead17b,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,533,48.0
5,6,content_fd16e3475c29,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,429,9.0
6,7,content_6476d1d8c050,client_19581e27de,2,WEAK_CTR_AND_STALE,REFRESH,0.00,313,304,67.8
7,8,content_4f241bad48a3,client_6208ef0f77,2,WEAK_CTR_AND_STALE,REFRESH,0.00,236,285,19.1
8,9,content_ea41fe5cf292,client_d029fa3a95,2,WEAK_CTR_AND_STALE,REFRESH,0.00,183,265,8.0
9,10,content_d25a099b3726,client_19581e27de,2,WEAK_CTR_AND_STALE,REFRESH,0.00,305,202,64.5


In [15]:
# Columns actually used by the baseline rule
baseline_features = [
    "ctr",
    "days_since_last_update",
    "impressions_90d"
]

# Columns that must NOT be used
forbidden_columns = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

print("Baseline features:")
print(baseline_features)

print("\nForbidden / label-derived columns:")
print(forbidden_columns)

print("\nLeakage check:")
print(
    "No forbidden columns are used in the baseline calculation."
)

Baseline features:
['ctr', 'days_since_last_update', 'impressions_90d']

Forbidden / label-derived columns:
['trend_pct', 'trend_direction', 'is_declining_label']

Leakage check:
No forbidden columns are used in the baseline calculation.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.